# ViGen AIO Studio - OmniVoice Colab Server

Notebook này khởi chạy máy chủ OmniVoice (Voice Cloning) trên Google Colab và kết nối với ứng dụng Desktop qua Cloudflare Tunnel.

### Hướng dẫn sử dụng:
1. **Thiết lập GPU**: Chọn **Runtime** -> **Change runtime type** -> chọn **T4 GPU**.
2. **Chạy Cell 1**: Cài đặt các thư viện cần thiết.
3. **Chạy Cell 2**: Khởi động máy chủ và lấy liên kết `trycloudflare.com` dán vào phần cài đặt của ứng dụng.

In [ ]:
#@title Bước 1: Cài đặt các thư viện cần thiết
!pip install flask omnivoice torch torchaudio

In [ ]:
#@title Bước 2: Khởi chạy máy chủ OmniVoice và Cloudflare Tunnel
import os
import re
import time
import tempfile
import subprocess
import threading
from flask import Flask, request, send_file, jsonify
import torch
from omnivoice import OmniVoice

app = Flask(__name__)
TEMP_DIR = tempfile.gettempdir()

print("Đang khởi tạo mô hình OmniVoice...")
device = "cuda:0" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if torch.cuda.is_available() else torch.float32
model = OmniVoice.from_pretrained("k2-fsa/OmniVoice", device_map=device, dtype=dtype)
print("Khởi tạo mô hình OmniVoice thành công!")

@app.route('/upload_ref', methods=['POST'])
def upload_ref():
    if 'file' not in request.files:
        return jsonify({"error": "No file part"}), 400
    file = request.files['file']
    if file.filename == '':
        return jsonify({"error": "No selected file"}), 400
    file_path = os.path.join(TEMP_DIR, f"omnivoice_ref_{os.urandom(4).hex()}.wav")
    file.save(file_path)
    return jsonify({"ref_path": file_path})

@app.route('/synthesize', methods=['POST'])
def synthesize():
    text = request.form.get('text', '')
    ref_path = request.form.get('ref_path', '')
    num_step = request.form.get('num_step', 32)
    guidance_scale = request.form.get('guidance_scale', 2.5)
    
    try:
        num_step = int(num_step)
        guidance_scale = float(guidance_scale)
    except ValueError:
        num_step = 32
        guidance_scale = 2.5
        
    if not text:
        return jsonify({"error": "No text provided"}), 400
        
    ref_audio_file = None
    if ref_path and os.path.exists(ref_path):
        ref_audio_file = ref_path
    elif 'file' in request.files:
        file = request.files['file']
        if file.filename != '':
            ref_audio_file = os.path.join(TEMP_DIR, f"omnivoice_ref_{os.urandom(4).hex()}.wav")
            file.save(ref_audio_file)
            
    if not ref_audio_file:
        return jsonify({"error": "No reference audio provided"}), 400
        
    output_path = os.path.join(TEMP_DIR, f"omnivoice_out_{os.urandom(4).hex()}.wav")
    try:
        import torchaudio
        print(f"[OmniVoice] Bắt đầu tổng hợp giọng nói: '{text[:50]}...'")
        
        # Tạo prompt clone giọng nói từ file mẫu
        voice_prompt = model.create_voice_clone_prompt(ref_audio=ref_audio_file, ref_text=None)
        
        # Thực hiện tổng hợp
        audio = model.generate(
            text=text,
            voice_clone_prompt=voice_prompt,
            language="vi"
        )
        
        # Chuyển đổi tensor/numpy array an toàn
        if torch.is_tensor(audio[0]):
            audio_tensor = audio[0].cpu()
        else:
            audio_tensor = torch.from_numpy(audio[0])
            
        # Đảm bảo định dạng 2D [channels, time] cho torchaudio
        if audio_tensor.ndim == 1:
            audio_tensor = audio_tensor.unsqueeze(0)
        elif audio_tensor.ndim == 2 and audio_tensor.shape[0] > audio_tensor.shape[1]:
            audio_tensor = audio_tensor.T
            
        # Lưu kết quả sử dụng torchaudio
        torchaudio.save(output_path, audio_tensor, model.sampling_rate)
        print(f"[OmniVoice] Hoàn thành! Đã lưu kết quả tại: {output_path}")
        return send_file(output_path, mimetype="audio/wav")
    except Exception as e:
        print(f"[OmniVoice] Lỗi: {str(e)}")
        return jsonify({"error": str(e)}), 500

def start_cloudflare_tunnel():
    print("\nĐang cài đặt Cloudflare Tunnel (cloudflared)...")
    subprocess.run(["wget", "-q", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb"])
    subprocess.run(["dpkg", "-i", "cloudflared-linux-amd64.deb"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    
    print("Đang khởi tạo đường truyền Cloudflare Tunnel...")
    proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", "http://127.0.0.1:39828"], 
        stdout=subprocess.PIPE, 
        stderr=subprocess.STDOUT, 
        text=True
    )
    
    tunnel_url = None
    for _ in range(30):
        time.sleep(1)
        line = proc.stdout.readline()
        if "trycloudflare.com" in line:
            match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
            if match:
                tunnel_url = match.group(0)
                print("\n" + "="*50)
                print(f" LIÊN KẾT ĐƯỜNG TRUYỀN GOOGLE COLAB CỦA BẠN:")
                print(f" {tunnel_url}")
                print("="*50 + "\n")
                print("Hãy sao chép liên kết trên dán vào tab Cấu Hình của ViGen AIO.")
                break

threading.Thread(target=start_cloudflare_tunnel, daemon=True).start()
app.run(port=39828, host='0.0.0.0')